In [8]:
print('hello')

hello


In [9]:
!uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.10.11 environment at: /Users/bandarusamanthuday/Desktop/llmops/.venv
Checked 6 packages in 56ms


In [10]:
import os
 
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')


## Data Ingestion in vector store


In [13]:
from langchain_community.document_loaders import TextLoader

/var/folders/kp/0lc6m80s5zg23d9kw2dypdr80000gn/T/ipykernel_24090/2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [16]:
loader=TextLoader("/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt",encoding='utf8')
documents=loader.load()

In [17]:
documents

[Document(metadata={'source': '/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt'}, page_content="Understanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:\n* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.\n* Autonomy: They can operate independently, making decisions and taking actions without continuous human oversight.\n* Perception: They can interpret information from their environment to inform their decisions.\n* Planning and Reasoning: They can formulate plans to reach

In [18]:
documents[0].page_content[:500] #first 500 characters

'Understanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to s'

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [21]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=200)

In [24]:
text_chunks=text_splitter.split_documents(documents)

In [25]:
text_chunks

[Document(metadata={'source': '/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': '/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': '/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt'}, page_content='AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined'),
 Document(metadata={'source': '/Users/bandarusamanthuday/Desktop/llmops/data/Agentic AI.txt'}, page_content='to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to op

In [26]:
!uv pip install faiss-cpu

Using Python 3.10.11 environment at: /Users/bandarusamanthuday/Desktop/llmops/.venv
Resolved 3 packages in 15.36s                                        
Prepared 1 package in 7.59s                                                  faiss-cpu            ------------------------------ 4.65 MiB/4.76 MiB           
Installed 1 package in 4ms                                  
 + faiss-cpu==1.15.1


In [28]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings

In [34]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/bandarusamanthuday/Desktop/llmops/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8660.53it/s]


In [35]:
vectorstore=FAISS.from_documents(text_chunks,embeddings)

In [36]:
vectorstore

In [37]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Understanding Agentic AI
--------------------------------------------------
Document 2:
Key Characteristics of Agentic AI
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 3:
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.
--------------------------------------------------
Document 4:
Applications of Agentic AI
The potential applications of agentic AI are vast and span across numerous industries.
--------------------------------------------------


In [38]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""


In [39]:
prompt=ChatPromptTemplate.from_template(template)

In [40]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [42]:
from langchain_core.output_parsers import StrOutputParser

In [43]:
output_parser=StrOutputParser()

In [50]:
retriever=vectorstore.as_retriever()

In [57]:
from langchain_openai import ChatOpenAI
llm_model=ChatOpenAI(model_name='gpt-4o-mini')

In [58]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

In [59]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [60]:
rag_chain.invoke("tell me about Agentic AI")

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}